# 00: Dataset Manifest Construction & Validation

This notebook demonstrates how to construct, validate, and serialize raw dataset manifests using the fluent `DatasetManifestBuilder` API and unified single-document YAML/JSON specifications.

### Context & Motivation
Before launching the `data-harmonize` ETL pipeline, all raw multi-source rasters (optical satellite imagery, digital elevation models, environmental domain masks, and categorical target labels) must be registered in a **dataset manifest**.

Rather than manually authoring error-prone JSON sidecars and copying file paths:
1. **Fluent Python API (`DatasetManifestBuilder`)**: Register continuous features, discrete domains, and categorical labels with automatic 1-based band mapping and reclassification schemes.
2. **Fail-Fast Upstream Validation**: Verifies raster contracts, band counts, categorical bounds, and scheme references in memory before initiating expensive ETL jobs.
3. **Single-Document Portability**: Serializes directly into clean, human-readable YAML (`manifest.yaml`) or machine-consumable JSON (`manifest.json`) with automatic relative path resolution.
4. **Seamless Pipeline Hand-off**: The materialized manifest feeds directly into `01_data_preparation.ipynb` for spatial world grid alignment and tiling.

## Environment Setup & Imports

In [ ]:
# If running in a cloud/remote notebook (e.g. Databricks, Colab),
# install the wheel:
# %pip install path/to/landseg-x.y.z-py3-none-any.whl

# Or if running from a local repo clone:
# %pip install -e ..

In [ ]:
# standard imports
import os
import pathlib
import sys
# third-party imports
import rasterio
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.geopipe.harmonize.manifest as manifest
    import landseg.testing as testing
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.geopipe.harmonize.manifest as manifest
    import landseg.testing as testing

print('Successfully imported landseg manifest tools!')

# Check if running in a Databricks environment
IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# Define the experiment root path where artifacts and results will be stored.
EXP_ROOT = pathlib.Path('./experiment')

if IS_DATABRICKS:
    # EXP_ROOT = pathlib.Path('/Volumes/my_catalog/my_schema/my_volume/experiment')
    print('Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.')

## Generate or Locate Raw Rasters

For local exploration and testing, we generate a synthetic multi-modal geospatial dataset under `experiment/input/` consisting of:
- **Sentinel-2 MSI** (10 continuous surface reflectance bands)
- **DEM** (1 continuous elevation band)
- **Geology & Soil** (discrete environmental domain masks)
- **Landcover & Lead Species** (multi-class categorical ground truth labels)

*(If working with real provincial or regional rasters, set your raster directory paths accordingly).*

In [ ]:
# generate dummy raw rasters for demonstration
testing.generate_pre_harmonization_data(EXP_ROOT / 'input')
raw_data_dir = EXP_ROOT / 'input' / 'raw_data'
print(f'Raw raster directory: {raw_data_dir.resolve()}')

## Inspect Raw GeoTIFF Metadata

Before registering a raster into the manifest, it is good practice to inspect its physical headers (CRS, band count, dimensions, pixel resolution, and nodata value) using `rasterio`.

In [ ]:
# inspect multi-band Sentinel-2 feature raster
s2_path = raw_data_dir / 'sample_sentinel2.tif'

with rasterio.open(s2_path) as ds:
    print('--- Sentinel-2 Header Inspection ---')
    print(f'Path: {s2_path.name}')
    print(f'CRS: {ds.crs}')
    print(f'Dimensions (W x H): {ds.width} x {ds.height}')
    print(f'Band Count: {ds.count}')
    print(f'Data Type: {ds.dtypes[0]}')
    print(f'NoData Value: {ds.nodata}')
    print(f'Resolution: ({ds.res[0]:.2f}m, {ds.res[1]:.2f}m)')

## Assemble Manifest with `DatasetManifestBuilder`

We now initialize `DatasetManifestBuilder` to programmatically register each raster layer using category-specific fluent methods.

In [ ]:
builder = manifest.DatasetManifestBuilder()

# 1. Register continuous feature raster (Sentinel-2 optical imagery)
# We can use the canonical SensorPreset enumeration or specify custom band names
builder.add_feature(
    name='sentinel2',
    path=raw_data_dir / 'sample_sentinel2.tif',
    bands=list(manifest.SensorPreset.SENTINEL_2.value),
    schemes={
        'rgb': ('blue', 'green', 'red'),
        'rgb_nir': ('blue', 'green', 'red', 'nir'),
        'surface': (
            'blue', 'green', 'red', 'red_edge1', 'red_edge2',
            'red_edge3', 'nir', 'narrow_nir', 'swir1', 'swir2'
        ),
    },
)

# 2. Register continuous feature raster (Digital Elevation Model)
builder.add_feature(
    name='dem',
    path=raw_data_dir / 'sample_dem.tif',
    bands=['dem'],
)

# 3. Register discrete domain knowledge (Geology & Soil)
builder.add_domain(
    name='mock_geology',
    path=raw_data_dir / 'sample_domain_1.tif',
    num_cls=4,
    index_base=1,
    ignore_cls=[255],
)

builder.add_domain(
    name='mock_soil',
    path=raw_data_dir / 'sample_domain_2.tif',
    num_cls=9,
    index_base=1,
    ignore_cls=[255],
)

# 4. Register discrete categorical ground truth label (Landcover)
builder.add_label(
    name='landcover',
    path=raw_data_dir / 'sample_landcover.tif',
    num_cls=2,
    index_base=1,
    ignore_cls=[255],
    classes={1: 'coniferous', 2: 'deciduous'},
    color_map={1: (34, 139, 34), 2: (218, 165, 32)},
    schemes={
        'binary': {
            'reclass': {'1': [1, 2]},
            'reclass_name': {'1': 'forest'},
        }
    },
)

# 5. Register discrete categorical ground truth label with taxonomy profile (Tree Species)
builder.add_label(
    name='leadspc',
    path=raw_data_dir / 'sample_leadspc.tif',
    num_cls=2,
    index_base=1,
    ignore_cls=[255],
    classes={1: 'SB_BLACK_SPRUCE', 2: 'PJ_JACK_PINE'},
    color_map={1: (0, 100, 0), 2: (107, 142, 35)},
    taxonomy_profile='ontario_tree_species_grouped_profiles',
)

print(f'Registered {len(builder.raw_entries)} raster layers into the builder.')

## Validate & Compile Manifest Entries

Invoking `.compile()` performs comprehensive validation:
- Enforces category integrity (`features`, `domains`, `labels`)
- Verifies physical raster presence on disk
- Validates contiguous 1-based band mappings
- Checks categorical class bounds and ignore-class ranges
- Audits scheme references against declared band names and target class IDs
- Computes SHA256 fingerprints to guard against duplicated layer configurations

In [ ]:
compiled_manifest = builder.compile()

print(f'Successfully compiled {len(compiled_manifest)} validated manifest entries:\n')
for entry in compiled_manifest:
    print(f"Layer: {entry['name']:<14} | Category: {entry['category']:<8} | Bands: {len(entry['band_mapping'])}")

## Materialize Manifest to Disk (JSON & YAML)

Calling `builder.materialize()` exports the verified manifest into both:
1. `manifest.json`: Single-document machine format used directly by data pipelines.
2. `manifest.yaml`: Clean, human-readable YAML specification with automatic relative path resolution for portability.

In [ ]:
# materialize manifest files into the raw data directory
builder.materialize(raw_data_dir, name='manifest')

json_manifest = raw_data_dir / 'manifest.json'
yaml_manifest = raw_data_dir / 'manifest.yaml'

print(f'Materialized JSON manifest: {json_manifest} ({json_manifest.stat().st_size} bytes)')
print(f'Materialized YAML manifest: {yaml_manifest} ({yaml_manifest.stat().st_size} bytes)\n')

# preview the first 35 lines of the generated YAML manifest
yaml_lines = yaml_manifest.read_text(encoding='utf-8').splitlines()
print('--- manifest.yaml preview ---')
print('\n'.join(yaml_lines[:35]))

## Declarative Workflow: Reload from YAML

The single-document YAML manifest is fully round-trip compatible. Teams can maintain `manifest.yaml` in version control, edit layers by hand, and reconstruct a verified builder using `DatasetManifestBuilder.from_file()`:

In [ ]:
# reconstruct builder from persisted YAML manifest
reloaded_builder = manifest.DatasetManifestBuilder.from_file(yaml_manifest)
recompiled_entries = reloaded_builder.compile()

print(f'Successfully reloaded and compiled {len(recompiled_entries)} layers from YAML!')
assert len(recompiled_entries) == len(compiled_manifest)

## Next Steps

Now that your dataset manifest has been constructed and validated:
1. Proceed to **`01_data_preparation.ipynb`** to generate the canonical World Grid, harmonize input rasters against the spatial canvas, and tile data blocks.
2. Reference the generated manifest directly via:
   ```python
   harmonize_configurator.set_dataset_manifest('experiment/input/raw_data/manifest.json')
   ```